# Aula 6 — Árvores de Decisão e Random Forest

Nesta prática, vamos classificar chamados de suporte como **urgentes (1)** ou **não urgentes (0)**. Os dados são sintéticos e servem apenas para aprendizagem.

**Objetivos:** treinar uma árvore, controlar sua complexidade, interpretar métricas e comparar com Random Forest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

## 1. Criar o conjunto de dados

Cada linha representa um chamado. As quatro primeiras colunas são características; `urgente` será o alvo. O `random_state` implícito na semente 42 torna o experimento reproduzível.

In [ ]:
rng = np.random.default_rng(42)
dados = pd.DataFrame({
    "usuarios_afetados": rng.integers(1, 201, 600),
    "risco_seguranca": rng.integers(0, 2, 600),
    "sistema_indisponivel": rng.integers(0, 2, 600),
    "impacto_financeiro": rng.integers(0, 2, 600),
})

dados["urgente"] = (
    (dados["risco_seguranca"] == 1)
    | ((dados["sistema_indisponivel"] == 1) & (dados["usuarios_afetados"] > 40))
    | ((dados["impacto_financeiro"] == 1) & (dados["usuarios_afetados"] > 20))
).astype(int)

# Inverte 10% dos rótulos para simular ruído e casos não observados.
indices = rng.choice(dados.index, size=60, replace=False)
dados.loc[indices, "urgente"] = 1 - dados.loc[indices, "urgente"]

dados.head()

In [ ]:
display(dados.describe())
display(dados["urgente"].value_counts().rename("quantidade"))
display(dados["urgente"].value_counts(normalize=True).rename("proporcao"))

## 2. Separar treino e teste

O modelo aprende com 75% dos dados. Os 25% restantes simulam exemplos ainda não vistos. `stratify=y` preserva aproximadamente a proporção das classes.

In [ ]:
X = dados.drop(columns="urgente")
y = dados["urgente"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

## 3. Baseline

O baseline prevê sempre a classe mais frequente. Ele é um ponto mínimo de comparação, não uma solução inteligente.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_treino, y_treino)

print("Acurácia do baseline no treino:", baseline.score(X_treino, y_treino))
print("Acurácia do baseline no teste:", baseline.score(X_teste, y_teste))

## 4. Árvore de decisão

Começamos com uma árvore pequena para manter as regras legíveis.

In [ ]:
arvore = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=10,
    random_state=42
)
arvore.fit(X_treino, y_treino)
previsoes_arvore = arvore.predict(X_teste)

print("Acurácia no treino:", arvore.score(X_treino, y_treino))
print("Acurácia no teste:", arvore.score(X_teste, y_teste))

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    arvore,
    feature_names=X.columns,
    class_names=["não urgente", "urgente"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.show()

### Experimento A — profundidade

Compare treino e teste. Uma distância grande entre os dois resultados pode indicar sobreajuste.

In [ ]:
resultados_profundidade = []
for profundidade in [1, 3, 5, None]:
    modelo = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
    modelo.fit(X_treino, y_treino)
    resultados_profundidade.append({
        "profundidade": str(profundidade),
        "treino": modelo.score(X_treino, y_treino),
        "teste": modelo.score(X_teste, y_teste),
        "diferenca": modelo.score(X_treino, y_treino) - modelo.score(X_teste, y_teste)
    })

pd.DataFrame(resultados_profundidade).round(3)

## 5. Avaliar classificadores

A função abaixo mostra acurácia, precisão, recall, F1 e matriz de confusão. Neste problema, a classe positiva é `1 = urgente`.

In [ ]:
def avaliar(nome, y_real, y_previsto):
    resultado = {
        "modelo": nome,
        "acuracia": accuracy_score(y_real, y_previsto),
        "precisao": precision_score(y_real, y_previsto),
        "recall": recall_score(y_real, y_previsto),
        "f1": f1_score(y_real, y_previsto),
    }
    display(pd.DataFrame([resultado]).round(3))
    ConfusionMatrixDisplay.from_predictions(
        y_real, y_previsto, display_labels=["não urgente", "urgente"]
    )
    plt.title(nome)
    plt.show()
    return resultado

resultado_arvore = avaliar("Árvore", y_teste, previsoes_arvore)

## 6. Random Forest

A floresta combina várias árvores treinadas com amostras e subconjuntos de características diferentes.

In [ ]:
floresta = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
floresta.fit(X_treino, y_treino)
previsoes_floresta = floresta.predict(X_teste)

resultado_floresta = avaliar("Random Forest", y_teste, previsoes_floresta)

In [ ]:
comparacao = pd.DataFrame([resultado_arvore, resultado_floresta]).set_index("modelo")
comparacao.round(3)

In [ ]:
importancias = pd.Series(
    floresta.feature_importances_, index=X.columns
).sort_values(ascending=True)

importancias.plot.barh(title="Importância das características na Random Forest")
plt.xlabel("Importância calculada pelo modelo")
plt.show()

## 7. Desafio da dupla

1. Treine **duas árvores** com combinações diferentes de `max_depth` e `min_samples_leaf`.
2. Registre acurácia de treino e teste, precisão, recall e F1.
3. Compare as árvores com a Random Forest.
4. Escolha um modelo. Justifique em três frases: **escolha**, **evidência** e **limitação**.

> Não escolha apenas pela maior acurácia. Considere o custo de deixar um chamado urgente passar.

In [ ]:
# Árvore A — execute e depois altere os valores
arvore_a = DecisionTreeClassifier(
    max_depth=2, min_samples_leaf=5, random_state=42
)

# Árvore B — execute e depois altere os valores
arvore_b = DecisionTreeClassifier(
    max_depth=5, min_samples_leaf=15, random_state=42
)

### Conclusão da dupla

- **Modelo escolhido:**
- **Evidências numéricas:**
- **Erro mais importante neste contexto:**
- **Limitação antes de usar em produção:**